In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchao.quantization import quantize_, Int4WeightOnlyConfig, IntxWeightOnlyConfig
from torchao.quantization.qat import QATConfig
from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e
from torch.utils.data import DataLoader, TensorDataset

In [15]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [16]:
from sklearn.model_selection import train_test_split


df = pd.read_csv('data/compas_carla.csv')

# scale numeric columns
numeric_cols = df.drop('score', axis=1).select_dtypes(include=[np.number]).columns
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# get categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns
df_final = pd.get_dummies(df, columns=categorical_cols, dtype=float)

X_train, X_test, y_train, y_test = train_test_split(df_final.drop('score', axis=1), df_final['score'], test_size=0.2, random_state=42)

X_train.head()


,age,two_year_recid,priors_count,length_of_stay,c_charge_degree_F,c_charge_degree_M,race_African-American,race_Other,sex_Female,sex_Male
4586,0.465942,-0.913929,-0.684413,-0.313191,0.0,1.0,0.0,1.0,0.0,1.0
5160,-0.812832,1.094177,-0.473593,-0.291773,1.0,0.0,0.0,1.0,0.0,1.0
4271,0.124935,1.094177,1.634606,-0.270355,0.0,1.0,0.0,1.0,0.0,1.0
978,-0.983335,-0.913929,-0.473593,-0.313191,0.0,1.0,0.0,1.0,0.0,1.0
5171,-0.471826,-0.913929,-0.684413,-0.334609,0.0,1.0,0.0,1.0,0.0,1.0


In [17]:
# Quantization-aware training (QAT) example

# For grouped Int4 quantization to work, the input features (in_features) of every 
# single nn.Linear layer you want to quantize must be perfectly divisible by 32.

# the first layer has 7 or 10 input features, which is not divisible by 32,
# so we can ignore quantizing the first layer.

# we can try later to add some dummy features, but that may not make sense for recourse???

class ModelQuant(nn.Module):
    def __init__(self, input_dim):
        super(ModelQuant, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

    def predict_proba(self, X):
        X = torch.from_numpy(np.array(X)).float()
        class1_probs = self.network(X).detach().numpy()
        # round probability to 0 or 1
        prediction = (class1_probs > 0.5).astype(int)
        return np.hstack((1-prediction, prediction))
        #return np.hstack((class0_probs,class1_probs))

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

In [18]:
# filters out layers that cannot be quantized, and applies quantization to the rest
def filter_divisible_by_32(module, fqn):
    return isinstance(module, nn.Linear) and module.in_features % 32 == 0

In [19]:
def evaluate(model, X, y):
    model.eval()
    with torch.no_grad():
        proba = model(X).squeeze()
        predicted = (proba >= 0.5).float()
        accuracy = (predicted == y).float().mean().item()
    return accuracy

In [20]:
# def train_model(model, dataloader, X_test, y_test, criterion, optimizer, num_epochs, quantized):
#     print("=" * 45)
#     print(f"Training {model.__class__.__name__} model...")
#     model.train()
#     if quantized:
#         print("Using quantization-aware training (QAT)")

#         model.qconfig = get_default_qconfig('fbgemm')
#         prepare_qat(model, inplace=True)
    
#     for epoch in range(num_epochs):
#         for inputs, targets in dataloader:
#             optimizer.zero_grad()
#             logits = model(inputs).squeeze()
#             loss = criterion(logits, targets)
#             loss.backward()
#             optimizer.step()
#             optimizer.zero_grad()

#         if (epoch + 1) % 5 == 0:
#             acc = evaluate(model, X_test, y_test)
#             print(f"Epoch {epoch+1:2d} | Loss: {loss.item():.4f} | Accuracy: {acc*100:.2f}%")

In [21]:
non_qat_model = ModelQuant(input_dim=10)

X_train_tensor = torch.tensor(X_train.to_numpy(), dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.to_numpy(), dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.float32)

train_data = TensorDataset(X_train_tensor, y_train_tensor)
test_data = TensorDataset(X_test_tensor, y_test_tensor)
dataloader = DataLoader(train_data, batch_size=32, shuffle=True)

criterion = nn.BCELoss()
optimizer_non_qat = optim.Adam(non_qat_model.parameters(), lr=0.001)


In [22]:
# Non-QAT training
print("=" * 45)
print(f"Training {non_qat_model.__class__.__name__} model...")
non_qat_model.train()

for epoch in range(40):
    for inputs, targets in dataloader:
        optimizer_non_qat.zero_grad()
        logits = non_qat_model(inputs).squeeze()
        loss = criterion(logits, targets)
        loss.backward()
        optimizer_non_qat.step()

    if (epoch + 1) % 5 == 0:
        acc = evaluate(non_qat_model, X_test_tensor, y_test_tensor)
        print(f"Epoch {epoch+1:2d} | Loss: {loss.item():.4f} | Accuracy: {acc*100:.2f}%")

fp32_acc = evaluate(non_qat_model, X_test_tensor, y_test_tensor)
print(f"FP32 Test Accuracy: {fp32_acc * 100:.2f}%")

Training ModelQuant model...
Epoch  5 | Loss: 0.3592 | Accuracy: 86.15%
Epoch 10 | Loss: 0.5505 | Accuracy: 85.34%
Epoch 15 | Loss: 0.5202 | Accuracy: 85.02%
Epoch 20 | Loss: 0.5898 | Accuracy: 86.15%
Epoch 25 | Loss: 0.1897 | Accuracy: 85.91%
Epoch 30 | Loss: 0.2386 | Accuracy: 86.07%
Epoch 35 | Loss: 0.1277 | Accuracy: 85.26%
Epoch 40 | Loss: 0.1690 | Accuracy: 83.16%
FP32 Test Accuracy: 83.16%


In [23]:
qat_model = ModelQuant(input_dim=10)

In [24]:
# QAT training
print("=" * 45)
print(f"Training {qat_model.__class__.__name__} model...")

base_config = Int4WeightOnlyConfig(group_size=32)
quantize_(qat_model, QATConfig(base_config, step='prepare'), filter_fn=filter_divisible_by_32)

optimizer_qat = optim.Adam(qat_model.parameters(), lr=0.001)

qat_model.train()

for epoch in range(40):
    for inputs, targets in dataloader:
        optimizer_qat.zero_grad()
        logits = qat_model(inputs).squeeze()
        loss = criterion(logits, targets)
        loss.backward()
        optimizer_qat.step()

    if (epoch + 1) % 5 == 0:
        acc = evaluate(qat_model, X_test_tensor, y_test_tensor)
        print(f"Epoch {epoch+1:2d} | Loss: {loss.item():.4f} | Accuracy: {acc*100:.2f}%")

INT4_acc = evaluate(qat_model, X_test_tensor, y_test_tensor)
print(f"INT4 Test Accuracy: {INT4_acc * 100:.2f}%")

Training ModelQuant model...
Epoch  5 | Loss: 0.2457 | Accuracy: 85.59%
Epoch 10 | Loss: 0.7014 | Accuracy: 85.43%
Epoch 15 | Loss: 0.1252 | Accuracy: 85.67%
Epoch 20 | Loss: 0.5131 | Accuracy: 85.83%
Epoch 25 | Loss: 0.1492 | Accuracy: 85.51%
Epoch 30 | Loss: 0.3152 | Accuracy: 85.26%
Epoch 35 | Loss: 0.2100 | Accuracy: 84.62%
Epoch 40 | Loss: 0.1278 | Accuracy: 84.70%
INT4 Test Accuracy: 84.70%


In [25]:
# 7. Compare model sizes
def model_size_kb(m):
    return sum(p.numel() * p.element_size() for p in m.parameters()) / 1024

In [26]:
# # 6. Convert QAT model to fully quantized INT8
qat_model.eval()
quantize_(qat_model, QATConfig(base_config, step='convert'), filter_fn=filter_divisible_by_32)
qat_size = model_size_kb(qat_model)

qat_acc = evaluate(qat_model, X_test_tensor, y_test_tensor)
print(f"\nQuantized INT8 Test Accuracy: {qat_acc * 100:.2f}%")


ImportError: Requires mslk >= 1.0.0

In [ ]:
fp32_size = model_size_kb(non_qat_model)
q_size    = model_size_kb(qat_model)

print("\n" + "=" * 45)
print(f"FP32  model size : {fp32_size:.2f} KB")
print(f"INT4  model size : {q_size:.2f} KB")
print(f"Size reduction   : {(1 - q_size/fp32_size)*100:.1f}%")
print(f"Accuracy delta   : {(qat_acc - fp32_acc)*100:+.2f}%")

# 8. Single prediction example
qat_model.eval()
with torch.no_grad():
    sample = X_test_tensor[1].unsqueeze(0)
    logit  = qat_model(sample).squeeze()
    prob   = torch.sigmoid(logit).item()
    pred   = 1 if prob >= 0.5 else 0
    actual = int(y_test_tensor[1].item())

print("\n" + "=" * 45)
print(f"Sample prediction → Prob: {prob:.4f} | Pred: {pred} | Actual: {actual}")


FP32  model size : 101.76 KB
INT4  model size : 101.76 KB
Size reduction   : 0.0%
Accuracy delta   : +1.38%

Sample prediction → Prob: 0.7298 | Pred: 1 | Actual: 1
